In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("spark_sql")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

df_green = spark.read.parquet("data/pq/green/*/*")
df_yellow = spark.read.parquet("data/pq/yellow/*/*")
print(len(df_green.columns), "green columns", len(df_yellow.columns), "yellow columns")



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/14 20:05:35 WARN Utils: Your hostname, DESKTOP-8DK3P4V, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/14 20:05:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 20:05:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/14 20:05:43 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.

20 green columns 18 yellow columns


In [3]:
df_green = (
    df_green
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
)
df_yellow = (
    df_yellow
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
)

In [4]:
yellow_columns = set(df_yellow.columns)
common_columns = [c for c in df_green.columns if c in yellow_columns]
print(len(common_columns), common_columns)

18 ['VendorID', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'payment_type', 'congestion_surcharge']


In [5]:
df_green_sel = df_green.select(common_columns).withColumn("service_type", F.lit("green"))
df_yellow_sel = df_yellow.select(common_columns).withColumn("service_type", F.lit("yellow"))

df_trips = df_green_sel.unionAll(df_yellow_sel)

In [6]:
df_trips.groupBy("service_type").count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [7]:
df_trips.filter("service_type" == "yellow").show()

PySparkTypeError: [NOT_EXPECTED_TYPE] Argument `condition` should be Column or str, got bool.

In [ ]:
df_trips.createOrReplaceTempView("trips_data")

spark.sql("""
    SELECT service_type, COUNT(*) AS trips
    FROM trips_data
    GROUP BY service_type
""").show()

+------------+--------+
|service_type|   trips|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



In [ ]:
sql_version = spark.sql("""
    SELECT service_type, COUNT(*) AS trips
    FROM trips_data
    GROUP BY service_type
""")
df_version = df_trips.groupBy("service_type").agg(F.count("*").alias("trips"))

print("=== SQL ===")
sql_version.explain()
print("=== DataFrame ===")
df_version.explain()

=== SQL ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[service_type#44], functions=[count(1)])
   +- Exchange hashpartitioning(service_type#44, 200), ENSURE_REQUIREMENTS, [plan_id=337]
      +- HashAggregate(keys=[service_type#44], functions=[partial_count(1)])
         +- Union
            :- Project [green AS service_type#44]
            :  +- FileScan parquet [] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(19 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/pq/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<>
            +- Project [yellow AS service_type#45]
               +- FileScan parquet [] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(19 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/pq/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<>


=== DataFrame ===
== Physical Plan ==
AdaptiveSparkPlan isF

In [9]:
df_revenue = spark.sql("""
            SELECT
        PULocationID AS revenue_zone,
        date_trunc('month', pickup_datetime) AS revenue_month,
        service_type,

        SUM(fare_amount) AS revenue_monthly_fare,
        SUM(extra) AS revenue_monthly_extra,
        SUM(mta_tax) AS revenue_monthly_mta_tax,
        SUM(tip_amount) AS revenue_monthly_tip_amount,
        SUM(tolls_amount) AS revenue_monthly_tolls_amount,
        SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
        SUM(total_amount) AS revenue_monthly_total_amount,
        SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

        AVG(passenger_count) AS avg_monthly_passenger_count,
        AVG(trip_distance) AS avg_monthly_trip_distance
    FROM trips_data
    GROUP BY 1, 2, 3
           """)

df_revenue.show()

{"ts": "2026-11-14 20:06:54.287", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `trips_data` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o28.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `trips_data` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a 

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `trips_data` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 18 pos 9;
'Aggregate [unresolvedordinal(1), unresolvedordinal(2), unresolvedordinal(3)], ['PULocationID AS revenue_zone#76, 'date_trunc(month, 'pickup_datetime) AS revenue_month#77, 'service_type, 'SUM('fare_amount) AS revenue_monthly_fare#78, 'SUM('extra) AS revenue_monthly_extra#79, 'SUM('mta_tax) AS revenue_monthly_mta_tax#80, 'SUM('tip_amount) AS revenue_monthly_tip_amount#81, 'SUM('tolls_amount) AS revenue_monthly_tolls_amount#82, 'SUM('improvement_surcharge) AS revenue_monthly_improvement_surcharge#83, 'SUM('t

In [8]:
df_revenue.coalesce(1).write.mode("overwrite").parquet("data/report/revenue/")

NameError: name 'df_revenue' is not defined